# 07 Coarse grid data (256³)

Builds 256³ versions of the TNG300-3 dark matter and subhalo fields (2×2×2 block means) and a 256³ T-web (tidal-tensor) mask recomputed from the dark matter with σ_eff = 1 h⁻¹ Mpc. Then it checks how much the coarser grid costs on its own: the 256³ mask, upsampled to 512³, is scored against the 512³ mask. That is the best any 256³ model could score at 512³.

Run once. Afterwards train `A256_scratch_L10` in notebook 05.

Runtime: CPU is enough, with **High-RAM**. A few minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Write the 256³ files

In [ ]:
!python /content/deepvoid/scripts/make_coarse.py

## 2. Ceiling: a perfect 256³ prediction scored at 512³

On the test slab only, as in training runs.

In [ ]:
import numpy as np
from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.resample import upsampled_argmax
from deepvoid.split import slab_split

catalog = Catalog()
fine = catalog.load('tng300-3/mask').data
coarse = catalog.load('tng300-3/n256/mask').data
onehot = np.eye(4, dtype=np.float32)[coarse]
up = upsampled_argmax(onehot, 2)
test = slab_split(512, buffer=16)['test'].mask(512)
s = scores(confusion_matrix(fine, up, mask=test))
from pathlib import Path
from deepvoid.report import Report
rep = Report(Path('/content/drive/MyDrive/deepvoid_runs/coarse_grid_check'))
rep.record(ceiling_256_at_512=s)
print({k: round(v, 3) for k, v in s.items() if k in ('mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1')})
print('class fractions 512:', np.round(np.bincount(fine.ravel(), minlength=4) / fine.size, 3))
print('class fractions 256:', np.round(np.bincount(coarse.ravel(), minlength=4) / coarse.size, 3))

## 3. 512³ and 256³ side by side

The same thin slice of the box in both grids (a 512³ slice is averaged over the two cells that make up one 256³ cell, so both show 0.8 h⁻¹ Mpc of depth). Density panels share a colour scale. The squares on the mask panels are a 96³ training crop: 38 h⁻¹ Mpc at 512³ (white) and 77 h⁻¹ Mpc at 256³ (black).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle

L = 205.0
J = 128                                  # slice index on the 256^3 grid; 512^3 cells 2J and 2J+1
CLASSES = ['void', 'wall', 'filament', 'halo']
cmap_mask = ListedColormap(['#2b6cb0', '#9ae6b4', '#f6ad55', '#c53030'])

dm512 = catalog.load('tng300-3/dm').data
dm256 = catalog.load('tng300-3/n256/dm').data
sh512 = catalog.load('tng300-3/subhalos/L10').data
sh256 = catalog.load('tng300-3/n256/subhalos/L10').data

def logd(x, mean):
    return np.log10(x / mean + 1e-2)

def slab512(a):
    return a[:, :, 2 * J:2 * J + 2].mean(-1)

rows = {
    '512³': [logd(slab512(dm512), dm512.mean()), logd(slab512(sh512), sh512.mean()), fine[:, :, 2 * J]],
    '256³': [logd(dm256[:, :, J], dm256.mean()), logd(sh256[:, :, J], sh256.mean()), coarse[:, :, J]],
}
titles = ['dark matter, log10(ρ/ρ̄ + 0.01)', 'subhalos λ = 10, log10(ρ/ρ̄ + 0.01)', 'T-web mask']
# Colour ranges from the non-empty cells: a sparse tracer field is mostly
# exact zeros, which would otherwise pin both percentiles to the floor.
def colour_range(img):
    filled = img[img > np.log10(1e-2) + 1e-6]
    return (np.log10(1e-2), np.percentile(filled, 99.5)) if filled.size else (-2, 1)
lims = [colour_range(rows['512³'][i]) for i in range(2)]

for name, a in (('dark matter 512³', dm512), ('dark matter 256³', dm256),
                ('subhalos λ=10 512³', sh512), ('subhalos λ=10 256³', sh256)):
    print(f'{name}: {100 * (a == 0).mean():.3f}% of cells empty, min {a.min():.3g}, max {a.max():.3g}')

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for r, (grid, panels) in enumerate(rows.items()):
    for c, img in enumerate(panels):
        ax = axes[r, c]
        if c < 2:
            im = ax.imshow(img.T, origin='lower', extent=[0, L, 0, L], cmap='magma',
                           vmin=lims[c][0], vmax=lims[c][1])
        else:
            im = ax.imshow(img.T, origin='lower', extent=[0, L, 0, L], cmap=cmap_mask,
                           vmin=-0.5, vmax=3.5, interpolation='nearest')
            for side, colour in ((96 * L / 512, 'white'), (96 * L / 256, 'black')):
                ax.add_patch(Rectangle((20, 20), side, side, fill=False, ec=colour, lw=2))
        ax.set_title(f'{grid}: {titles[c]}')
        ax.set_xlabel('x [h⁻¹ Mpc]'); ax.set_ylabel('y [h⁻¹ Mpc]')
cbar = fig.colorbar(im, ax=axes[:, 2], ticks=range(4), shrink=0.6)
cbar.ax.set_yticklabels(CLASSES)
fig.suptitle('TNG300-3: the same slice at 512³ (top) and 256³ (bottom)')
rep.show(fig, '3_slices')

## 4. Close-up around the densest spot in the slice

40 h⁻¹ Mpc across. Left: the 512³ mask. Middle: the 256³ mask blown up to 512³, as a 256³ model's predictions would be scored. Right: cells where the two disagree, coloured by the true (512³) class; small halos are where most of the loss is.

In [ ]:
half = 50                                # 50 cells at 512^3 = 20 h^-1 Mpc
sl = dm512[:, :, 2 * J]
i0, j0 = np.unravel_index(np.argmax(sl), sl.shape)
win = (np.arange(i0 - half, i0 + half) % 512, np.arange(j0 - half, j0 + half) % 512)

a = fine[:, :, 2 * J][np.ix_(*win)]
b = up[:, :, 2 * J][np.ix_(*win)]
diff = np.where(a != b, a, np.nan)
ext = [0, 2 * half * L / 512] * 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, title in zip(axes, (a, b, diff),
                          ('512³ mask', '256³ mask upsampled to 512³', 'disagreement (true class shown)')):
    ax.imshow(img.T, origin='lower', extent=ext, cmap=cmap_mask, vmin=-0.5, vmax=3.5,
              interpolation='nearest')
    ax.set_title(title); ax.set_xlabel('[h⁻¹ Mpc]')
axes[2].set_facecolor('#eeeeee')
fig.suptitle('40 h⁻¹ Mpc close-up: 512³ mask vs 256³ mask upsampled')
rep.show(fig, '4_closeup')

## 5. Where each 512³ class ends up at 256³

Each row is a true (512³) class on the test slab; the columns are the class the upsampled 256³ mask gives those cells. The diagonal is what survives the coarser grid.

In [ ]:
cm = confusion_matrix(fine, up, mask=test).astype(float)
frac = cm / cm.sum(1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(frac, cmap='Blues', vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{frac[i, j]:.2f}', ha='center', va='center',
                color='white' if frac[i, j] > 0.5 else 'black')
ax.set_xticks(range(4), CLASSES); ax.set_yticks(range(4), CLASSES)
ax.set_title('Where each 512³ class ends up at 256³ (test slab)')
ax.set_xlabel('256³ (upsampled)'); ax.set_ylabel('512³ (true)')
rep.show(fig, '5_class_transfer')
rep.record(class_transfer=frac)

Figures and numbers are saved in `deepvoid_runs/coarse_grid_check/`. For comparison, the λ = 10 subhalo model trained at 512³ from scratch scores a Matthews correlation coefficient (MCC) of ≈ 0.63, so a ceiling well above that means the coarse grid itself is not the limit.